In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

from rdkit import Chem

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



c:\Users\shiva\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\shiva\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

Seed: 42


In [4]:
DATA_PATH = "../data/processed/drug_protein_dataset.csv"

merged_df = pd.read_csv(DATA_PATH)

print("Shape:", merged_df.shape)
print()
print(merged_df.columns.tolist())

Shape: (98805, 11)

['Ligand SMILES', 'Target Name', 'Ki (nM)', 'pKi', 'Protein_ID', 'Description', 'Sequence', 'Length', 'Tokens', 'InputIDs', 'Target_Name']


In [5]:
sample_size = min(5000, len(merged_df))

sample_df = merged_df.sample(
    sample_size,
    random_state=SEED
).reset_index(drop=True)

print("Sample shape:", sample_df.shape)

Sample shape: (5000, 11)


In [6]:
required_columns = [
    "Ligand SMILES",
    "Sequence",
    "pKi"
]

missing = [
    col for col in required_columns
    if col not in sample_df.columns
]

print("Missing columns:", missing)

Missing columns: []


In [7]:
sample_df = sample_df.dropna(
    subset=[
        "Ligand SMILES",
        "Sequence",
        "pKi"
    ]
).reset_index(drop=True)

print("After removing missing values:", sample_df.shape)

After removing missing values: (5000, 11)


In [8]:
AA_VOCAB = {
    "A": 0,
    "C": 1,
    "D": 2,
    "E": 3,
    "F": 4,
    "G": 5,
    "H": 6,
    "I": 7,
    "K": 8,
    "L": 9,
    "M": 10,
    "N": 11,
    "P": 12,
    "Q": 13,
    "R": 14,
    "S": 15,
    "T": 16,
    "V": 17,
    "W": 18,
    "Y": 19
}

print("Amino acids:", len(AA_VOCAB))

Amino acids: 20


In [9]:
MAX_PROTEIN_LENGTH = 512

def encode_protein(seq):
    seq = str(seq).upper()

    tokens = [
        AA_VOCAB[a]
        for a in seq
        if a in AA_VOCAB
    ]

    tokens = tokens[:MAX_PROTEIN_LENGTH]

    if len(tokens) < MAX_PROTEIN_LENGTH:
        tokens += [0] * (
            MAX_PROTEIN_LENGTH - len(tokens)
        )

    return torch.tensor(
        tokens,
        dtype=torch.long
    )

In [10]:
protein = encode_protein(
    sample_df.iloc[0]["Sequence"]
)

print(protein.shape)
print(protein[:20])

torch.Size([512])
tensor([10,  7,  8, 15, 15, 18, 14,  8,  7,  0, 10,  9,  0,  0,  0, 17, 12,  9,
         9,  9])


In [11]:
def smiles_to_graph(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    x = []

    for atom in mol.GetAtoms():

        x.append([
            atom.GetAtomicNum() / 100.0
        ])

    edge_index = []

    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        edge_index.append([i, j])
        edge_index.append([j, i])

    x = torch.tensor(
        x,
        dtype=torch.float
    )

    if len(edge_index) == 0:

        edge_index = torch.empty(
            (2, 0),
            dtype=torch.long
        )

    else:

        edge_index = torch.tensor(
            edge_index,
            dtype=torch.long
        ).t().contiguous()

    return Data(
        x=x,
        edge_index=edge_index
    )

In [12]:
test_graph = smiles_to_graph("CCO")

print(test_graph)
print("Nodes:", test_graph.num_nodes)
print("Edges:", test_graph.edge_index.shape)

Data(x=[3, 1], edge_index=[2, 4])
Nodes: 3
Edges: torch.Size([2, 4])


In [13]:
valid_rows = []
bad_rows = []

for i in range(len(sample_df)):

    graph = smiles_to_graph(
        sample_df.iloc[i]["Ligand SMILES"]
    )

    if graph is not None:
        valid_rows.append(i)
    else:
        bad_rows.append(i)

clean_df = sample_df.iloc[
    valid_rows
].reset_index(drop=True)

print("Bad samples:", len(bad_rows))
print("Valid samples:", len(clean_df))
print("Shape:", clean_df.shape)

Bad samples: 6
Valid samples: 4994
Shape: (4994, 11)


In [14]:
class DrugProteinDataset(Dataset):

    def __init__(self, dataframe):

        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        graph = smiles_to_graph(
            row["Ligand SMILES"]
        )

        protein = encode_protein(
            row["Sequence"]
        )

        label = torch.tensor(
            float(row["pKi"]),
            dtype=torch.float
        )

        return graph, protein, label

In [15]:
dataset = DrugProteinDataset(clean_df)

print("Dataset size:", len(dataset))

Dataset size: 4994


In [16]:
graph, protein, label = dataset[0]

print(graph)
print()
print("Protein:", protein.shape)
print()
print("Label:", label)

Data(x=[20, 1], edge_index=[2, 46])

Protein: torch.Size([512])

Label: tensor(2.8239)


In [17]:
indices = np.arange(len(dataset))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED
)

print("Training samples:", len(train_idx))
print("Testing samples:", len(test_idx))

Training samples: 3995
Testing samples: 999


In [18]:
def collate_fn(batch):

    graphs = [item[0] for item in batch]

    proteins = torch.stack([
        item[1]
        for item in batch
    ])

    labels = torch.stack([
        item[2]
        for item in batch
    ])

    graph_batch = Batch.from_data_list(
        graphs
    )

    return (
        graph_batch,
        proteins,
        labels
    )

In [21]:
train_dataset = torch.utils.data.Subset(
    dataset,
    train_idx
)

test_dataset = torch.utils.data.Subset(
    dataset,
    test_idx
)

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

print("DataLoaders ready")

DataLoaders ready


In [22]:
graph_batch, protein_batch, labels = next(
    iter(train_loader)
)

print(graph_batch)
print()
print("Protein batch:", protein_batch.shape)
print()
print("Labels:", labels.shape)

DataBatch(x=[841, 1], edge_index=[2, 1810], batch=[841], ptr=[33])

Protein batch: torch.Size([32, 512])

Labels: torch.Size([32])


In [23]:
class DrugEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = GCNConv(
            1,
            64
        )

        self.conv2 = GCNConv(
            64,
            128
        )

    def forward(
        self,
        graph
    ):

        x = graph.x
        edge_index = graph.edge_index

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        x = F.relu(x)

        x = global_mean_pool(
            x,
            graph.batch
        )

        return x

In [24]:
class ProteinEncoder(nn.Module):

    def __init__(
        self,
        vocab_size=20,
        embedding_dim=128,
        num_layers=2,
        nhead=8
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=nhead,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, protein):

        x = self.embedding(protein)

        x = self.transformer(x)

        x = x.mean(dim=1)

        return x

In [25]:
class PharmaGPT(nn.Module):

    def __init__(self):

        super().__init__()

        self.drug_encoder = DrugEncoder()

        self.protein_encoder = ProteinEncoder()

        self.head = nn.Sequential(

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(
                128,
                1
            )
        )

    def forward(
        self,
        graph,
        protein
    ):

        drug_embedding = self.drug_encoder(
            graph
        )

        protein_embedding = self.protein_encoder(
            protein
        )

        fused = torch.cat(
            [
                drug_embedding,
                protein_embedding
            ],
            dim=1
        )

        output = self.head(
            fused
        )

        return output

In [26]:
model = PharmaGPT().to(device)

print(model)

PharmaGPT(
  (drug_encoder): DrugEncoder(
    (conv1): GCNConv(1, 64)
    (conv2): GCNConv(64, 128)
  )
  (protein_encoder): ProteinEncoder(
    (embedding): Embedding(20, 128)
    (transformer): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (head): Sequential(
    (0): Linear(in_features=256, o

In [27]:
graph_batch = graph_batch.to(device)
protein_batch = protein_batch.to(device)
labels = labels.to(device)

prediction = model(
    graph_batch,
    protein_batch
)

print("Prediction shape:", prediction.shape)
print(prediction[:5])

Prediction shape: torch.Size([32, 1])
tensor([[0.0371],
        [0.0649],
        [0.0911],
        [0.0842],
        [0.0696]], grad_fn=<SliceBackward0>)


In [28]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

print("Optimizer ready")


Optimizer ready


In [29]:
def train_one_epoch():

    model.train()

    total_loss = 0.0

    for graph, protein, labels in train_loader:

        graph = graph.to(device)
        protein = protein.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        predictions = model(
            graph,
            protein
        ).squeeze(1)

        loss = criterion(
            predictions,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item() * labels.size(0)
        )

    return total_loss / len(train_loader.dataset)

In [30]:
def evaluate():

    model.eval()

    predictions = []
    actuals = []

    with torch.no_grad():

        for graph, protein, labels in test_loader:

            graph = graph.to(device)
            protein = protein.to(device)

            output = model(
                graph,
                protein
            ).squeeze(1)

            predictions.extend(
                output.cpu().numpy()
            )

            actuals.extend(
                labels.numpy()
            )

    predictions = np.array(predictions)
    actuals = np.array(actuals)

    mae = mean_absolute_error(
        actuals,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            actuals,
            predictions
        )
    )

    r2 = r2_score(
        actuals,
        predictions
    )

    return mae, rmse, r2

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):

    train_loss = train_one_epoch()

    mae, rmse, r2 = evaluate()

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"MAE: {mae:.4f} | "
        f"RMSE: {rmse:.4f} | "
        f"R²: {r2:.4f}"
    )

Epoch 1/5 | Loss: 14.8655 | MAE: 1.2614 | RMSE: 1.6062 | R²: -0.0401
Epoch 2/5 | Loss: 2.6254 | MAE: 1.2568 | RMSE: 1.6023 | R²: -0.0350


In [ ]:
checkpoint_dir = "../models/checkpoints"

os.makedirs(
    checkpoint_dir,
    exist_ok=True
)

checkpoint_path = (
    checkpoint_dir
    + "/pharmagpt_fusion_v1.pt"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epochs": EPOCHS,
        "seed": SEED
    },
    checkpoint_path
)

print("Model saved:")
print(checkpoint_path)

In [ ]:
model.eval()

graph, protein, actual = dataset[0]

graph = Batch.from_data_list(
    [graph]
).to(device)

protein = protein.unsqueeze(0).to(device)

with torch.no_grad():

    prediction = model(
        graph,
        protein
    )

print("Actual pKi:", actual.item())
print("Predicted pKi:", prediction.item())